In [179]:
import sys, os
from omegaconf import OmegaConf
import torch

sys.path.append("/home/jgershon/git/cleo")
from optimization_util import get_feasible_mask, get_seqs_from_action
from ensemble import Ensemble


config_path = "/home/jgershon/git/cleo/config/momi_acqf_opt.yaml"
cfg = OmegaConf.load(config_path)

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [180]:
# load surrogate model
surrogate_ckpt_path = os.path.join(cfg.surrogate_ckpt, "last.ckpt")
surrogate_config_path = os.path.join(cfg.surrogate_ckpt, "config.yaml")
surrogate_config = OmegaConf.load(surrogate_config_path)


ckpt = torch.load(surrogate_ckpt_path, map_location=DEVICE)
model = Ensemble(surrogate_config)
model.load_state_dict(ckpt["state_dict"])
model = model.to(DEVICE)
print("Loaded surrogate model from", cfg.surrogate_ckpt)

# load fragment dictionary
with open(cfg.opt_loop.fragment_dictionary, "r") as f:
    fragment_dictionary = json.load(f)
fragment_dictionary = {int(k):v for k,v in fragment_dictionary.items()}


Loaded surrogate model from /home/jgershon/git/cleo/ckpt/momi_mlp_no_val/momi_mlp_no_val.2025-12-15-22-18-23


In [ ]:
# 250611 - adding new features
class BatchUCBwithEntropy:
    def __init__(self, model, model_batch_size=64, gamma=0.1, eps=1e-8, sequence_wise=True):
        """
        Args:
            model (Ensemble): The ensemble model to use for predictions.
            model_batch_size (int): Batch size for model inference.
            gamma (float): Entropy weight for the acquisition function.
        """
        self.model = model # Ensemble model
        self.model_batch_size = model_batch_size # batch size for model inference
        self.gamma = gamma # entropy weight
        self.eps = eps # small value to avoid log(0)
        self.sequence_wise = sequence_wise # compute rewards per sequence rather than for whole batch

    @torch.no_grad()
    def __call__(self, X):

        N, q, d = X.shape
        X_r = X.reshape(N*q, d) # Flatten first two dimensions for batching
        r = X_r.shape[0]

        num_batches = r // self.model_batch_size + (1 if r % self.model_batch_size > 0 else 0)
        if r % self.model_batch_size > 0:
            num_batches += 1

        ucb_list = []
        for i in range(num_batches):
            start = i * self.model_batch_size
            end = min((i + 1) * self.model_batch_size, r)
            batch_X = X_r[start:end]

            # get predictions
            out = self.model(batch_X)

            # calculate UCB
            ucb = out['mu'] + out['sigma']
            ucb_list.append(ucb)


        if self.sequence_wise:
            ucb_per_seq = torch.cat(ucb_list, dim=0).reshape(N, q)


            # calculate how unique each sequence is
            X_seq_view = X.view(N, q, -1, 20)
            L = X_seq_view.shape[2]
            seq_flat = X_seq_view.view(N, q, L*20)
            seq_similiarities = torch.matmul(seq_flat, seq_flat.transpose(-1, -2)) / L
            seq_similiarities = seq_similiarities.mean(dim=-1) # mean over final dim 

            reward = ucb_per_seq + self.gamma * (1-seq_similiarities)

            metrics = {
                'ucb': ucb_per_seq.mean().item(),
                'seq_similarity': seq_similiarities.mean().item()
            }

        else:
            # rebatch the outputs so N, q
            batched_ucb = torch.cat(ucb_list, dim=0).reshape(N, q).mean(dim=1)  # Average over q dim so just (N,)

            # compute residue wise entropy
            X_seq_view = X.view(N, q, -1, 20)
            X_freqs = X_seq_view.sum(dim=1)/q # sum over batch to get frequencies at each position
            entropy = (-torch.sum(X_freqs * torch.log(X_freqs + self.eps), dim=-1)).mean(dim=-1)  # Entropy calculation over the sequence length

            reward = batched_ucb + self.gamma * entropy # (N,)

            metrics = {
                'ucb': batched_ucb.mean().item(),
                'entropy': entropy.mean().item(),
            }

        

        return reward, metrics


# create acqf 
acqf = BatchUCBwithEntropy(
    model, 
    model_batch_size=cfg.acqf.model_batch_size, 
    gamma=cfg.acqf.gamma, 
    eps=cfg.acqf.eps
)

In [236]:
# run optimization loop
q = cfg.opt_loop.q
lr = cfg.opt_loop.lr
N = cfg.opt_loop.N
num_iter = cfg.opt_loop.num_iter
device = DEVICE

feasible_mask = get_feasible_mask(fragment_dictionary)

# initialize policy
policy = torch.randn(q, feasible_mask.shape[0], feasible_mask.shape[1])*0.001 #scaling factor
feasible_mask = (feasible_mask[None]).repeat(q,1,1)

# set non fragments to << 0 so they do not get sampled
policy[~feasible_mask] = -torch.inf
policy = policy.to(device)
policy = torch.nan_to_num(policy)
policy = policy.requires_grad_(True)


optimizer = torch.optim.Adam([policy], lr=lr)
collected_rewards = []

metric_logs = {"step":[], "reward":[]}

# REINFORCE LOOP
with tqdm(total=num_iter, desc='Reward: ') as pbar:

    for i in range(num_iter):
        
        optimizer.zero_grad()

        # softmax policy to get probabilities

        soft_policy = torch.softmax(policy,dim=-1)

        # make categorical distribution
        m = torch.distributions.Categorical(soft_policy)

        sampled_actions = []
        sampled_log_probs = []
        for j in range(N):
            # sample action
            action = m.sample()
            sampled_log_probs.append(m.log_prob(action)[None])
            sampled_actions.append(action[None])

        actions = torch.cat(sampled_actions,dim=0)
        log_probs = torch.cat(sampled_log_probs,dim=0)

        # convert from fragments to num seq
        sampled_seqs = get_seqs_from_action(actions, fragment_dictionary)[0].long()
        sampled_seqs = torch.nn.functional.one_hot(sampled_seqs, num_classes = 20)
        sampled_seqs = sampled_seqs.reshape(sampled_seqs.shape[0],sampled_seqs.shape[1],-1)

        # get reward
        reward, metrics = acqf(sampled_seqs.to(device))

        # log metrics
        for k, v in metrics.items():
            if k not in metric_logs:
                metric_logs[k] = []
            metric_logs[k].append(v)
        metric_logs["step"].append(i)
        metric_logs["reward"].append(float(reward.mean().detach()))

        # save logs
        # if i % 50 == 0:
        #     metrics_df = pd.DataFrame(metric_logs)
        #     metrics_path = os.path.join(out_path, "metrics.csv")
        #     metrics_df.to_csv(metrics_path, index=False)


        if reward.shape[0] == N and reward.shape[1] == q:
            # reward is provided at sequence level
            # compute group relative advantage estimate (GRPO style)
            adv = (reward - reward.mean(dim=1).unsqueeze(-1))/(reward.std(dim=1).unsqueeze(-1))

            loss = (-log_probs.sum(dim=2) * adv).sum(dim=1).mean()
        
        else:
            # substract beta term "moving average"
            beta_term = 0
            if i > 0:
                beta_term = np.mean(collected_rewards)

            collected_rewards.append(float(reward.mean().detach()))

            beta_subtract_reward = reward - beta_term

            # calculate policy loss
            loss = (-log_probs * beta_subtract_reward[...,None,None].repeat(1,q,actions.shape[-1])).sum(dim=(1,2)).mean()

        # opt
        loss.backward()
        optimizer.step()

        pbar.set_postfix({'Reward': f'{float(reward.mean()):.3f}'})
        pbar.update(1)

        

Reward:   0%|          | 0/25000 [00:00<?, ?it/s]

Reward:   0%|          | 6/25000 [01:22<95:50:08, 13.80s/it, Reward=-0.668]


KeyboardInterrupt: 

In [231]:
# compute group relative advantage
loss

tensor(6.2313e-05, grad_fn=<MeanBackward0>)

In [233]:
loss

tensor(-0.0073, grad_fn=<MeanBackward0>)

In [235]:
reward.mean()

tensor(-0.6670)

In [223]:
reward.shape

torch.Size([12, 384])

In [238]:
one_hot_seq

tensor([[[[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          ...,
          [0, 0, 0,  ..., 0, 1, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [1, 0, 0,  ..., 0, 0, 0]],

         [[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          ...,
          [0, 0, 0,  ..., 0, 1, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [1, 0, 0,  ..., 0, 0, 0]],

         [[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 1, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          ...,
          [0, 0, 0,  ..., 0, 1, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [1, 0, 0,  ..., 0, 0, 0]],

         ...,

         [[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          ...,
          [0, 0, 0,  ..., 0, 1, 0],
          [0, 0, 0,  ..., 0, 0, 0],
          [1, 0, 0,  ..., 0, 0, 0]],

         [[0, 0, 0,  ..., 0, 0, 0],
          [0, 0, 

In [ ]:
# x_flat = x.view(B, N, L * 20).float()      # [B, N, D]
# matches = torch.matmul(x_flat, x_flat.transpose(-1, -2))  # [B, N, N]
# hamming = L - matches



In [ ]:
reward = ucb_vals + 

torch.Size([12, 384])

In [202]:
ucb_vals.shape

torch.Size([12, 384])